# 1. DART 데이터 수집
사업보고서 원문을 DART API로 수집 후 Azure Blob Storage에 저장

In [ ]:
import os
import re
import io
import time
import zipfile
import requests
import xml.etree.ElementTree as ET
from dotenv import load_dotenv
from azure.storage.blob import BlobServiceClient

load_dotenv()

DART_API_KEY = os.getenv('DART_API_KEY')
STORAGE_CONNECTION_STRING = os.getenv('AZURE_STORAGE_CONNECTION_STRING')
CONTAINER_NAME = os.getenv('AZURE_STORAGE_CONTAINER_NAME')

print('환경변수 로드 완료')

## 수집 대상 기업 설정

In [ ]:
TARGET_COMPANIES = [
    '삼성전자',
    'SK하이닉스',
    '현대자동차',
    'NAVER',
    '카카오',
]

TARGET_YEAR = 2023

## corp_code 조회

In [ ]:
def get_corp_code(company_name: str) -> str | None:
    url = 'https://opendart.fss.or.kr/api/corpCode.xml'
    params = {'crtfc_key': DART_API_KEY}

    response = requests.get(url, params=params)
    zip_file = zipfile.ZipFile(io.BytesIO(response.content))
    xml_content = zip_file.read('CORPCODE.xml')

    root = ET.fromstring(xml_content)
    for corp in root.findall('list'):
        name = corp.findtext('corp_name', '')
        code = corp.findtext('corp_code', '')
        stock = corp.findtext('stock_code', '')
        if name == company_name and stock:
            return code
    return None

# 테스트
code = get_corp_code('삼성전자')
print(f'삼성전자 corp_code: {code}')

## 사업보고서 접수번호 조회

In [ ]:
def get_report_no(corp_code: str, year: int = 2023) -> str | None:
    url = 'https://opendart.fss.or.kr/api/list.json'
    params = {
        'crtfc_key': DART_API_KEY,
        'corp_code': corp_code,
        'bgn_de': f'{year}0101',
        'end_de': f'{year}1231',
        'pblntf_ty': 'A',
        'page_count': 10,
    }

    response = requests.get(url, params=params)
    data = response.json()

    if data.get('status') != '000':
        print(f'  API 오류: {data.get("message")}')
        return None

    for item in data.get('list', []):
        if '사업보고서' in item.get('report_nm', ''):
            return item.get('rcept_no')
    return None

# 테스트
rcept_no = get_report_no(code, year=TARGET_YEAR)
print(f'삼성전자 접수번호: {rcept_no}')

## 원문 다운로드 및 텍스트 추출

In [ ]:
def extract_text(content: str) -> str:
    td_texts = re.findall(r'<TD[^>]*>(.*?)</TD>', content, re.DOTALL)
    if td_texts:
        content = ' '.join(td_texts)
    text = re.sub(r'<[^>]+>', ' ', content)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()


def get_document(rcept_no: str) -> str | None:
    url = 'https://opendart.fss.or.kr/api/document.xml'
    params = {
        'crtfc_key': DART_API_KEY,
        'rcept_no': rcept_no,
    }

    response = requests.get(url, params=params)
    if response.status_code != 200:
        return None

    try:
        zip_file = zipfile.ZipFile(io.BytesIO(response.content))
    except zipfile.BadZipFile:
        return None

    full_text = ''
    for name in zip_file.namelist():
        if name.endswith('.xml'):
            content = zip_file.read(name).decode('utf-8', errors='ignore')
            text = re.sub(r'<[^>]+>', ' ', content)
            text = re.sub(r'\s+', ' ', text).strip()
            full_text += text + '\n'

    return full_text if full_text else None

# 테스트
text = get_document(rcept_no)
print(f'추출 텍스트 길이: {len(text):,}자')
print('\n--- 앞부분 미리보기 ---')
print(text[:500])

## Azure Blob Storage 업로드

In [ ]:
def upload_to_blob(company_name: str, text: str):
    blob_service = BlobServiceClient.from_connection_string(STORAGE_CONNECTION_STRING)
    container = blob_service.get_container_client(CONTAINER_NAME)

    if not container.exists():
        container.create_container()
        print(f'컨테이너 생성: {CONTAINER_NAME}')

    blob_name = f'{company_name}_{TARGET_YEAR}.txt'
    blob_client = container.get_blob_client(blob_name)
    blob_client.upload_blob(text.encode('utf-8'), overwrite=True)
    print(f'  업로드 완료: {blob_name} ({len(text):,}자)')

## 전체 수집 실행

In [ ]:
for company in TARGET_COMPANIES:
    print(f'[{company}] 처리 중...')

    corp_code = get_corp_code(company)
    if not corp_code:
        print(f'  corp_code 조회 실패 - 건너뜀\n')
        continue

    rcept_no = get_report_no(corp_code, year=TARGET_YEAR)
    if not rcept_no:
        print(f'  사업보고서 없음 - 건너뜀\n')
        continue

    print(f'  접수번호: {rcept_no}')

    text = get_document(rcept_no)
    if not text:
        print(f'  원문 추출 실패 - 건너뜀\n')
        continue

    upload_to_blob(company, text)
    print(f'  완료\n')

    time.sleep(1)

print('=== 수집 완료 ===')